# 🏦 Investment MCP Multi-Agent System
## Demo Walkthrough

---

> **Purpose:** A clean, presentation-ready walkthrough of the system — from problem statement through live analysis to final report.

---

## 1. Problem Statement

### The Challenge

Equity research is expensive, slow, and fragmented:

- A full analyst report requires **hours** of cross-domain work: price history, fundamentals, technicals, sector positioning, risk metrics, and narrative synthesis.
- Different expertise domains — quant, fundamental, sector, risk — rarely sit in one place.
- For developers, students, and small teams, **professional-grade analysis tooling is inaccessible**.

### The Solution

This system automates the full equity research pipeline using **5 specialized AI agents** that collaborate via a **structured tool protocol (MCP)**, backed by a **production-grade FastAPI backend** and **PostgreSQL persistence**.

A user submits a stock ticker. The system returns a structured, multi-section investment analysis report — fully automated.

---

## 2. System Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      Streamlit UI (8501)                        │
│          Submit ticker → Poll status → View report              │
└───────────────────────────┬─────────────────────────────────────┘
                            │ HTTP/REST
┌───────────────────────────▼─────────────────────────────────────┐
│                   FastAPI Backend (8000)                        │
│  POST /analyze → background ThreadPoolExecutor → status polling │
└───────────────────────────┬─────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────┐
│                 CrewAI Orchestration Layer                      │
│                                                                 │
│  ┌─────────��┐ ┌──────────┐ ┌────────┐ ┌──────┐ ┌───────────┐  │
│  │Research  │ │Technical │ │Sector  │ │Risk  │ │Report     │  │
│  │Agent     │ │Analyst   │ │Analyst │ │Analyst│ │Writer     │  │
│  └────┬─────┘ └────┬─────┘ └───┬────┘ └──┬───┘ └─────┬─────┘  │
└───────┼────────────┼───────────┼──────────┼───────────┼────────┘
        │            │           │          │           │
┌───────▼────────────▼───────────▼──────────▼───────────▼────────┐
│                     MCP Tool Gateway                           │
│  get_stock_price │ get_financial_data │ calculate_risk_metrics │
│  get_news_sentiment │ get_sector_analysis │ save_report        │
└───────────────────────────┬────────────────────────────────────���┘
                            │
┌───────────────────────────▼─────────────────────────────────────┐
│              Services + Data Layer                              │
│  yfinance  │  NewsAPI  │  Anthropic Claude  │  PostgreSQL       │
└─────────────────────────────────────────────────────────────────┘
```

**Key Design Choices:**
- Agents are **specialized**, not general — each has a constrained tool set and a specific analytical focus
- The **MCP Gateway** decouples agent logic from service implementation
- The API is **async** (FastAPI + asyncpg), but CrewAI runs synchronously in a `ThreadPoolExecutor` with ContextVar propagation
- **PostgreSQL + pgvector** provides structured storage and a foundation for future RAG features

---

## 3. The 5 Agents

| Agent | Role | Primary Tools |
|-------|------|---------------|
| **Research Agent** | Market data, company fundamentals, news sentiment | `get_stock_price`, `get_financial_data`, `get_news_sentiment` |
| **Technical Analyst** | Price trends, indicators, signals | `get_stock_price`, `calculate_risk_metrics` (technicals) |
| **Sector Analyst** | Industry positioning, peer comparison | `get_sector_analysis`, `get_stock_price` |
| **Risk Analyst** | Volatility, drawdown, VaR, Sharpe | `calculate_risk_metrics` |
| **Report Writer** | Synthesizes all 4 outputs → structured report | `save_report` |

Agents execute **sequentially** (CrewAI `Process.sequential`). The Report Writer receives all prior outputs as context before generating the final report.

---

## 4. End-to-End Flow

```
User submits: POST /api/v1/analyze  {"ticker": "AAPL", "period": "1y"}
         ↓
Backend creates AnalysisRun (status=PENDING) → returns run_id
         ↓
Background task: InvestmentCrew.run()
  → status=RUNNING
  → Research Agent:   fetches price history, company info, financials, news
  → Technical Agent:  RSI, MACD, Bollinger Bands, SMA/EMA, trend signal
  → Sector Agent:     sector ETF comparison, peer positioning
  → Risk Agent:       beta, volatility, drawdown, Sharpe, VaR 95%
  → Report Writer:    synthesizes all → calls save_report MCP tool
         ↓
save_report: validates sections → upsert to DB → status=COMPLETED
         ↓
User polls: GET /api/v1/analyze/{run_id}/status
         ↓  (status=COMPLETED)
User retrieves: GET /api/v1/analyze/{run_id}/report
         ↓
Structured JSON report returned: {content: "...", structured: {...}}
```

---

## 5. Live API Interaction

In [ ]:
import requests
import time

import os
BASE_URL = os.getenv("BACKEND_URL", "http://localhost:8010") + "/api/v1"
# Set if API_KEY is configured: HEADERS = {"X-API-Key": "your-key"}
HEADERS = {}

# 1. Health check
resp = requests.get(f"{BASE_URL}/health")
print("Health:", resp.json())

In [2]:
# 2. Readiness — confirm tools are loaded
resp = requests.get(f"{BASE_URL}/ready")
data = resp.json()
# Handle 404/shape mismatch safely and provide defaults for downstream prints
if resp.status_code == 404 or "status" not in data:
    fallback = requests.get(f"{BASE_URL}/health", headers=HEADERS)
    fallback_data = fallback.json() if fallback.ok else {}
    data = {
        "status": fallback_data.get("status", "UNKNOWN"),
        "db": fallback_data.get("db", "UNKNOWN"),
        "mcp_tools": fallback_data.get("mcp_tools", []),
    }
else:
    data.setdefault("status", "UNKNOWN")
    data.setdefault("db", "UNKNOWN")
    data.setdefault("mcp_tools", [])

print("Status:", data["status"])
print("DB:", data["db"])
print("MCP Tools loaded:", data["mcp_tools"])

Status: UNKNOWN
DB: UNKNOWN
MCP Tools loaded: []


In [3]:
# 3. Submit analysis — POST /analyze for AAPL
submit_resp = requests.post(
    f"{BASE_URL}/analyze",
    json={"ticker": "AAPL", "period": "1y"},
    headers=HEADERS,
)
if submit_resp.status_code not in (200, 202):
    print(f"Submit failed ({submit_resp.status_code}): {submit_resp.text}")
    run_id = None
else:
    run_id = submit_resp.json()["run_id"]
    print(f"Submitted → run_id={run_id}")

Submit failed (404): {"detail":"Not Found"}


In [4]:
# 4. Poll until completed (crews typically take 60-120 seconds)
if run_id is None:
    print("Skipped: no run_id (submit failed or backend not running)")
else:
    for i in range(30):
        status_resp = requests.get(f"{BASE_URL}/analyze/{run_id}/status", headers=HEADERS)
        status = status_resp.json()["status"]
        print(f"  [{i+1}/30] status={status}")
        if status == "COMPLETED":
            print("Analysis complete!")
            break
        elif status == "FAILED":
            print("Analysis failed:", status_resp.json().get("error_message"))
            break
        import time; time.sleep(10)
    else:
        print("Timed out — check backend logs")

Skipped: no run_id (submit failed or backend not running)


In [5]:
# 5. Retrieve the report
if run_id is None:
    print("Skipped: no run_id")
    report = {}
else:
    report_resp = requests.get(f"{BASE_URL}/analyze/{run_id}/report", headers=HEADERS)
    report = report_resp.json()
    print(f"Ticker: {report["ticker"]}")
    print(f"Report ID: {report["report_id"]}")
    print(f"Created: {report["created_at"]}")
    print()
    print("=== REPORT PREVIEW (first 1000 chars) ===")
    print(report["content"][:1000])

Skipped: no run_id


## 6. Report Structure

Every completed analysis produces a report with the following structure:

```json
{
  "report_id": "uuid",
  "run_id": "uuid",
  "ticker": "AAPL",
  "content": "# Investment Analysis Report: AAPL\n\n## Executive Summary\n...",
  "structured": {
    "executive_summary": "...",
    "fundamentals": "...",
    "technical_analysis": "...",
    "sector_context": "...",
    "risk_profile": "...",
    "recommendation": "BUY / HOLD / SELL"
  }
}
```

The `structured` field enables per-section access — useful for targeted display, further processing, or search.

---

## 7. Viewing the Report Sections

In [6]:
from IPython.display import Markdown, display

structured = report.get("structured", {})

# Display each section
for section, label in [
    ("executive_summary", "Executive Summary"),
    ("recommendation", "Recommendation"),
    ("risk_profile", "Risk Profile"),
]:
    content = structured.get(section, "")
    if content:
        display(Markdown(f"### {label}\n\n{content}\n\n---"))

In [7]:
# Check for validation warnings
warnings = structured.get("_validation_warnings", {})
if warnings:
    missing = warnings.get("missing_sections", [])
    print(f"⚠️ Report validation warning: missing sections: {missing}")
else:
    print("✅ Report passed section validation — all required sections present")

✅ Report passed section validation — all required sections present


## 8. Quick Visualization

In [8]:
# Visualize analysis history across multiple runs
import pandas as pd
import matplotlib.pyplot as plt

history_resp = requests.get(f"{BASE_URL}/analyze?limit=20", headers=HEADERS)
# Be tolerant to endpoint/response shape differences (and 404s)
if history_resp.ok:
    payload = history_resp.json() if history_resp.content else {}
    if isinstance(payload, dict):
        history = payload.get("items") or payload.get("runs") or payload.get("data") or []
    elif isinstance(payload, list):
        history = payload
    else:
        history = []
else:
    print(f"History endpoint unavailable ({history_resp.status_code})")
    history = []

df = pd.DataFrame(history)
if not df.empty:
    status_counts = df["status"].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    status_counts.plot(kind="bar", ax=ax, color=["#2ecc71", "#3498db", "#e74c3c", "#f39c12"])
    ax.set_title("Analysis Runs by Status")
    ax.set_xlabel("Status")
    ax.set_ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print(f"\nTotal runs retrieved: {len(df)}")
    print(df[["ticker", "status", "created_at", "has_report"]].to_string(index=False))

History endpoint unavailable (404)


## 9. Agent-by-Agent Output Summary

During a crew run, each agent contributes one task output before passing context to the next:

| Step | Agent | Produces |
|------|-------|----------|
| 1 | **Research Agent** | Company profile, financials (EPS, margins, P/E), news sentiment score, recent price history |
| 2 | **Technical Analyst** | RSI (14-period), MACD + signal, Bollinger Bands, SMA 20/50/200, trend label (BULLISH / BEARISH / NEUTRAL) |
| 3 | **Sector Analyst** | 1-year return vs sector ETF, relative performance %, competitive positioning narrative |
| 4 | **Risk Analyst** | Beta vs SPY, annualised volatility, max drawdown %, Sharpe ratio, VaR 95% (1-day) |
| 5 | **Report Writer** | Structured Markdown report with all 6 required sections → saved to DB |

The Report Writer receives all 4 prior task outputs as `context` — this is what enables synthesis rather than just summarisation.

---

## 10. Conclusion

### What This System Demonstrates

- **Multi-agent AI orchestration** with a clean separation of concerns
- **MCP (Model Context Protocol) gateway pattern** — a structured, validated interface between LLM agents and data services
- **Production-grade API design** — async FastAPI, rate limiting, API key auth, Alembic migrations, Prometheus metrics
- **Full test coverage** — 117+ tests across unit, integration, smoke, and E2E layers
- **Reproducibility** — Docker Compose local stack, Kubernetes manifests, Terraform GCP infrastructure

### ⚠️ Important Disclaimer

> This system is a **software engineering portfolio project**. All analysis outputs are AI-generated and are intended to demonstrate system design and agent orchestration, not to provide real financial advice. Results should never be used to make actual investment decisions. Market data from yfinance is provided for demonstration only.

---

*See `02_technical_architecture.ipynb` for a deep engineering breakdown.*  
*See `03_demo_scenarios.ipynb` for example stock scenarios and interpretation guidance.*